# Notebook 4 – Target Engineering

Objective

• Define customer decline based on future purchasing behavior.

• Calculate future revenue and revenue change metrics.

• Create the target variable for machine learning models.

Notebook 4 Plan

Step 1

Create: future_90d orders

Step 2

Calculate: future_revenue

Step 3

Calculate: current_revenue from recent 90 days

Step 4

Create: revenue_change_pct

Step 5

Create: decline_label

Step 6
Audit class balance, decline_label.value_counts()

This is the most important audit in the project because it tells us whether the problem is learnable.

In [ ]:
# ====================================================
# MOUNT GOOGLE DRIVE
# ====================================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ====================================================
# IMPORT LIBRARIES
# ====================================================

import pandas as pd
import numpy as np
# ====================================================
# LOAD INPUT DATASETS
# ====================================================

retail = pd.read_pickle(
    "/content/drive/MyDrive/Project files/retail_clean.pkl"
)

snapshot_df = pd.read_pickle(
    "/content/drive/MyDrive/Project files/snapshot_df.pkl"
)

snapshot_features = pd.read_pickle(
    "/content/drive/MyDrive/Project files/snapshot_features.pkl"
)

print(retail.shape)
print(snapshot_df.shape)
print(snapshot_features.shape)

(805549, 10)
(46098, 2)
(46098, 19)


In [ ]:
# ====================================================
# CREATE INVOICE LEVEL DATASET
# ====================================================

invoice_level = (
    retail
    .groupby(
        ["Customer ID", "Invoice", "InvoiceDate"],
        as_index=False
    )
    .agg(
        Revenue=("Revenue", "sum")
    )
)

print(invoice_level.shape)
invoice_level.head()

(37033, 4)


,Customer ID,Invoice,InvoiceDate,Revenue
0,12346,491725,2009-12-14 08:34:00,45.0
1,12346,491742,2009-12-14 11:00:00,22.5
2,12346,491744,2009-12-14 11:02:00,22.5
3,12346,492718,2009-12-18 10:47:00,22.5
4,12346,492722,2009-12-18 10:55:00,1.0


In [ ]:
#Step 1 — Create Future Transactions Table
# ====================================================
# CREATE FUTURE ORDERS WINDOW
# ====================================================

# Merge snapshots with all invoice-level orders

future_merge = snapshot_df.merge(
    invoice_level,
    on="Customer ID",
    how="inner"
)

print("Future Merge Shape:")
print(future_merge.shape)

Future Merge Shape:
(415975, 5)


In [ ]:
#Step 2 — Keep Only Future 90-Day Orders
# ====================================================
# FUTURE 90-DAY ORDERS
# ====================================================

future_90d = future_merge[
    (
        future_merge["InvoiceDate"]
        > future_merge["snapshot_date"]
    )
    &
    (
        future_merge["InvoiceDate"]
        <= (
            future_merge["snapshot_date"]
            + pd.Timedelta(days=90)
        )
    )
].copy()

print("Future 90-Day Shape:")
print(future_90d.shape)

future_90d.head()

Future 90-Day Shape:
(45220, 5)


,Customer ID,snapshot_date,Invoice,InvoiceDate,Revenue
18,12490,2010-05-31,515948,2010-07-15 15:51:00,509.00
45,12682,2010-05-31,510683,2010-06-03 09:29:00,447.02
46,12682,2010-05-31,511096,2010-06-06 15:49:00,573.86
47,12682,2010-05-31,513809,2010-06-29 09:28:00,322.24
48,12682,2010-05-31,515945,2010-07-15 15:32:00,398.54


In [ ]:
#Step 3 — Calculate Future Revenue
# ====================================================
# FUTURE REVENUE
# ====================================================

future_revenue = (
    future_90d
    .groupby(
        ["Customer ID", "snapshot_date"]
    )["Revenue"]
    .sum()
    .reset_index()
)

future_revenue.rename(
    columns={
        "Revenue": "future_revenue"
    },
    inplace=True
)

future_revenue.head()

,Customer ID,snapshot_date,future_revenue
0,12346,2010-10-31,77183.60
1,12346,2010-11-30,77183.60
2,12346,2010-12-31,77183.60
3,12347,2011-04-30,382.52
4,12347,2011-05-31,967.43


In [ ]:
#Step 4 — Merge Into Feature Dataset
# ====================================================
# ADD FUTURE REVENUE
# ====================================================

model_df = snapshot_features.merge(
    future_revenue,
    on=["Customer ID", "snapshot_date"],
    how="left"
)

model_df["future_revenue"] = (
    model_df["future_revenue"]
    .fillna(0)
)

model_df.head()

,Customer ID,snapshot_date,recency_days,frequency_180d,monetary_180d,aov_180d,previous_spend,recent_spend,spend_trend,previous_orders,recent_orders,frequency_trend,previous_aov,recent_aov,aov_trend,first_purchase_date,customer_age_days,lifetime_orders,lifetime_spend,future_revenue
0,12362,2010-05-31,180.0,0.0,0.00,0.000000,0.00,0.00,1.000000,0.0,0.0,1.0,0.00,0.000000,1.000000,2009-12-01 10:10:00,180,1,130.00,0.00
1,12490,2010-05-31,23.0,6.0,1943.23,323.871667,905.52,1037.71,1.145982,3.0,3.0,1.0,301.84,345.903333,1.145982,2009-12-01 12:52:00,180,7,2547.17,509.00
2,12533,2010-05-31,69.0,1.0,437.97,437.970000,0.00,437.97,1.000000,0.0,1.0,1.0,0.00,437.970000,1.000000,2009-12-01 11:50:00,180,2,1367.89,0.00
3,12636,2010-05-31,180.0,0.0,0.00,0.000000,0.00,0.00,1.000000,0.0,0.0,1.0,0.00,0.000000,1.000000,2009-12-01 09:55:00,180,1,141.00,0.00
4,12682,2010-05-31,24.0,9.0,5325.20,591.688889,3178.62,2146.58,0.675318,6.0,3.0,0.5,529.77,715.526667,1.350636,2009-12-01 09:28:00,180,10,5751.50,2200.33


In [ ]:
# ====================================================
# ADD CURRENT REVENUE
# ====================================================

model_df["current_revenue"] = (
    model_df["recent_spend"]
)

In [ ]:
model_df[
    [
        "current_revenue",
        "future_revenue"
    ]
].describe()

,current_revenue,future_revenue
count,46098.000000,46098.000000
mean,508.274197,526.858953
std,2524.597165,2864.837513
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,403.950000,393.730000
max,111637.440000,117634.530000


In [ ]:
model_df.shape

(46098, 21)

In [ ]:
# Current revenue equals zero

print(
    (model_df["current_revenue"] == 0).sum()
)

26618


In [ ]:
# Future revenue greater than zero

print(
    (model_df["future_revenue"] > 0).sum()
)

18839


In [ ]:
# Current = 0 and Future > 0

print(
    (
        (model_df["current_revenue"] == 0)
        &
        (model_df["future_revenue"] > 0)
    ).sum()
)

6237


In [ ]:
# Current > 0 and Future = 0

print(
    (
        (model_df["current_revenue"] > 0)
        &
        (model_df["future_revenue"] == 0)
    ).sum()
)

6878


In [ ]:
# ====================================================
# TYPE D CUSTOMERS
# Current Revenue = 0
# Future Revenue = 0
# ====================================================

print(
    (
        (model_df["current_revenue"] == 0)
        &
        (model_df["future_revenue"] == 0)
    ).sum()
)

20381


### Customer Type Analysis

* Total Snapshots = **46,098**

#### Snapshot Distribution

* Current Revenue = 0 → **26,618**
* Current Revenue = 0 & Future Revenue > 0 → **6,237**
* Current Revenue = 0 & Future Revenue = 0 → **20,381**
* Current Revenue > 0 & Future Revenue = 0 → **6,878**

#### Key Finding

* **20,381 snapshots** have:

  * Current Revenue = 0
  * Future Revenue = 0

* These customers were already inactive during the observation period and remained inactive during the prediction period.

#### Recommendation

* Remove snapshots where:

  `current_revenue = 0`

#### Reason

* The project objective is to **predict revenue decline among active customers**.
* Customers with zero current revenue are already inactive and do not provide information about revenue decline.
* Removing these records allows the model to focus on customers who are currently generating revenue and may decline in the future.

#### Impact

* Reduces noise in the dataset.
* Creates a more business-relevant prediction problem.
* Improves the model's ability to identify at-risk active customers.


In [ ]:
#Step 1 — Create Active-Customer Dataset
# ====================================================
# KEEP ONLY ACTIVE CUSTOMERS
# ====================================================

model_df_active = model_df[
    model_df["current_revenue"] > 0
].copy()

print("Active Dataset Shape:")
print(model_df_active.shape)

Active Dataset Shape:
(19480, 21)


In [ ]:
#Step 2 — Create Revenue Change %
# ====================================================
# REVENUE CHANGE %
# ====================================================

model_df_active["revenue_change_pct"] = (
    (
        model_df_active["future_revenue"]
        -
        model_df_active["current_revenue"]
    )
    /
    model_df_active["current_revenue"]
)

In [ ]:
#Step 3 — Audit Revenue Change
model_df_active["revenue_change_pct"].describe()

,revenue_change_pct
count,19480.000000
mean,0.053300
std,5.868951
min,-1.000000
25%,-1.000000
50%,-0.447671
75%,0.241144
max,367.541176


In [ ]:
# ====================================================
# DECLINE LABEL
# ====================================================

model_df_active["decline_label"] = np.where(
    model_df_active["revenue_change_pct"] <= -0.40,
    1,
    0
)

In [ ]:
#Audit Class Balance
model_df_active["decline_label"].value_counts()

,count
decline_label,
1,10092
0,9388


In [ ]:
model_df_active["decline_label"].value_counts(normalize=True) * 100

,proportion
decline_label,
1,51.806982
0,48.193018


In [ ]:
# ====================================================
# SAVE FINAL MODEL DATASET
# ====================================================

model_df_active.to_csv(
    "customer_decline_model_dataset.csv",
    index=False
)

print(model_df_active.shape)

(19480, 23)


Dataset:
Online Retail II

Unit of Analysis:
Customer Monthly Snapshot

Snapshots:
46,098

Active Snapshots Used:
19,480

Features:
10

Target: Revenue Decline ≥ 40%
within next 90 days

Class Balance: Decline = 51.8%

No Decline = 48.2%

In [ ]:
model_df_active

,Customer ID,snapshot_date,recency_days,frequency_180d,monetary_180d,aov_180d,previous_spend,recent_spend,spend_trend,previous_orders,...,recent_aov,aov_trend,first_purchase_date,customer_age_days,lifetime_orders,lifetime_spend,future_revenue,current_revenue,revenue_change_pct,decline_label
1,12490,2010-05-31,23.0,6.0,1943.23,323.871667,905.52,1037.71,1.145982,3.0,...,345.903333,1.145982,2009-12-01 12:52:00,180,7,2547.17,509.00,1037.71,-0.509497,1
2,12533,2010-05-31,69.0,1.0,437.97,437.970000,0.00,437.97,1.000000,0.0,...,437.970000,1.000000,2009-12-01 11:50:00,180,2,1367.89,0.00,437.97,-1.000000,1
4,12682,2010-05-31,24.0,9.0,5325.20,591.688889,3178.62,2146.58,0.675318,6.0,...,715.526667,1.350636,2009-12-01 09:28:00,180,10,5751.50,2200.33,2146.58,0.025040,0
5,12758,2010-05-31,31.0,2.0,1970.91,985.455000,0.00,1970.91,1.000000,0.0,...,985.455000,1.000000,2009-12-01 14:40:00,180,3,4425.59,1590.11,1970.91,-0.193210,0
6,12836,2010-05-31,40.0,4.0,1438.44,359.610000,600.35,838.09,1.396002,2.0,...,419.045000,1.396002,2009-12-01 14:19:00,180,5,1862.31,937.45,838.09,0.118555,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46076,18262,2011-08-31,39.0,1.0,149.48,149.480000,0.00,149.48,1.000000,0.0,...,149.480000,1.000000,2010-04-16 09:05:00,501,3,577.54,0.00,149.48,-1.000000,1
46080,18268,2011-08-31,33.0,1.0,25.50,25.500000,0.00,25.50,1.000000,0.0,...,25.500000,1.000000,2009-12-11 10:27:00,627,6,1490.23,0.00,25.50,-1.000000,1
46084,18272,2011-08-31,12.0,4.0,2106.45,526.612500,980.54,1125.91,1.148255,2.0,...,562.955000,1.148255,2010-03-04 12:53:00,544,7,3413.85,604.25,1125.91,-0.463323,1
46092,18281,2011-08-31,79.0,1.0,80.82,80.820000,0.00,80.82,1.000000,0.0,...,80.820000,1.000000,2010-05-11 10:49:00,476,2,201.14,0.00,80.82,-1.000000,1


In [ ]:
model_df_active.columns.tolist()

['Customer ID',
 'snapshot_date',
 'recency_days',
 'frequency_180d',
 'monetary_180d',
 'aov_180d',
 'previous_spend',
 'recent_spend',
 'spend_trend',
 'previous_orders',
 'recent_orders',
 'frequency_trend',
 'previous_aov',
 'recent_aov',
 'aov_trend',
 'first_purchase_date',
 'customer_age_days',
 'lifetime_orders',
 'lifetime_spend',
 'future_revenue',
 'current_revenue',
 'revenue_change_pct',
 'decline_label']

In [ ]:

# Remove duplicate spend columns

model_df_active = model_df_active.drop(
    columns=[
        "recent_spend_x",
        "recent_spend_y"
    ]
)

print(model_df_active.shape)

KeyError: "['recent_spend_x', 'recent_spend_y'] not found in axis"

In [ ]:
feature_cols = [
    "recency_days",
    "frequency_180d",
    "monetary_180d",
    "aov_180d",

    "previous_spend",
    "spend_trend",

    "previous_orders",
    "recent_orders",
    "frequency_trend",

    "previous_aov",
    "recent_aov",
    "aov_trend",

    "customer_age_days",
    "lifetime_orders",
    "lifetime_spend"
]

X = model_df_active[feature_cols]

y = model_df_active["decline_label"]

print(X.shape)
print(y.shape)

(19480, 15)
(19480,)


Final Modeling Dataset
Observations
19,480 active customer snapshots
Features
15
Target
decline_label

1 = Revenue decline ≥ 40%

0 = No significant decline

Class Balance
Decline     : 51.8%,
No Decline  : 48.2%

Nearly perfectly balanced.

In [ ]:
# ====================================================
# SAVE MODEL DATASET
# ====================================================

model_df_active.to_pickle(
    "/content/drive/MyDrive/Project files/model_df_active.pkl"
)

print("model_df_active saved successfully")

model_df_active saved successfully


In [ ]:
model_df_active.shape

model_df_active.head()


,Customer ID,snapshot_date,recency_days,frequency_180d,monetary_180d,aov_180d,previous_spend,recent_spend,spend_trend,previous_orders,...,recent_aov,aov_trend,first_purchase_date,customer_age_days,lifetime_orders,lifetime_spend,future_revenue,current_revenue,revenue_change_pct,decline_label
1,12490,2010-05-31,23.0,6.0,1943.23,323.871667,905.52,1037.71,1.145982,3.0,...,345.903333,1.145982,2009-12-01 12:52:00,180,7,2547.17,509.00,1037.71,-0.509497,1
2,12533,2010-05-31,69.0,1.0,437.97,437.970000,0.00,437.97,1.000000,0.0,...,437.970000,1.000000,2009-12-01 11:50:00,180,2,1367.89,0.00,437.97,-1.000000,1
4,12682,2010-05-31,24.0,9.0,5325.20,591.688889,3178.62,2146.58,0.675318,6.0,...,715.526667,1.350636,2009-12-01 09:28:00,180,10,5751.50,2200.33,2146.58,0.025040,0
5,12758,2010-05-31,31.0,2.0,1970.91,985.455000,0.00,1970.91,1.000000,0.0,...,985.455000,1.000000,2009-12-01 14:40:00,180,3,4425.59,1590.11,1970.91,-0.193210,0
6,12836,2010-05-31,40.0,4.0,1438.44,359.610000,600.35,838.09,1.396002,2.0,...,419.045000,1.396002,2009-12-01 14:19:00,180,5,1862.31,937.45,838.09,0.118555,0
